# prep_cedi_nidd_gbdt

Bereitet `train_80.csv` für das GBDT-Notebook vor.

In [2]:
import pandas as pd
from pathlib import Path

INPUT_CSV = Path("pcap_ml_output/final/test_20.csv")
OUTPUT_CSV = Path("pcap_ml_output/final/test_20_prepped_gbdt.csv")
LABEL_SOURCE_COL = "label_family"
LABEL_TARGET_COL = "Attack Type"

df = pd.read_csv(INPUT_CSV)
print("Input shape:", df.shape)
print("Input columns:", list(df.columns))

Input shape: (18911, 21)
Input columns: ['pkt_len_mean', 'pkt_len_std', 'pkt_len_min', 'pkt_len_median', 'iat_mean', 'iat_std', 'iat_min', 'iat_max', 'iat_median', 'tcp_ratio', 'udp_ratio', 'icmp_ratio', 'syn_ratio', 'ack_ratio', 'sport_missing_ratio', 'sport_high_ratio', 'unique_proto_count', 'unique_icmp_type_count', 'label_full', 'label_family', 'file_name']


In [3]:
def normalize_label(x: str) -> str:
    x = str(x)
    for suffix in ("_r0", "_r1"):
        if x.endswith(suffix):
            return x[:-3]
    return x

df[LABEL_TARGET_COL] = df[LABEL_SOURCE_COL].map(normalize_label)
print(df[[LABEL_SOURCE_COL, LABEL_TARGET_COL]].drop_duplicates().sort_values(LABEL_SOURCE_COL).to_string(index=False))

  label_family Attack Type
     gaming_r0      gaming
     gaming_r1      gaming
   n2_flooding n2_flooding
       n3_icmp     n3_icmp
        n3_tcp      n3_tcp
        n3_udp      n3_udp
   n4_flooding n4_flooding
   n6_flooding n6_flooding
       ping_r0        ping
       ping_r1        ping
  streaming_r0   streaming
  streaming_r1   streaming
webbrowsing_r0 webbrowsing
webbrowsing_r1 webbrowsing


In [4]:
drop_cols = [c for c in ["label_full", "label_family", "file_name"] if c in df.columns]
df = df.drop(columns=drop_cols)

for col in df.columns:
    if col != LABEL_TARGET_COL:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.replace([float("inf"), float("-inf")], pd.NA).fillna(0)

print("Output shape:", df.shape)
print("Class counts:")
print(df[LABEL_TARGET_COL].value_counts().sort_index())

Output shape: (18911, 19)
Class counts:
Attack Type
gaming         3585
n2_flooding    2000
n3_icmp        2000
n3_tcp         1475
n3_udp         2000
n4_flooding    1762
n6_flooding    2000
ping             82
streaming      4000
webbrowsing       7
Name: count, dtype: int64


In [5]:
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved to: {OUTPUT_CSV.resolve()}")

Saved to: /home/ubuntu/5g-nidd/cedi-nidd/pcap_ml_output/final/test_20_prepped_gbdt.csv


In [ ]:
# Quick check for your training notebook
X = df.drop(columns=[LABEL_TARGET_COL])
y = df[LABEL_TARGET_COL]
print("X shape:", X.shape)
print("y classes:", y.nunique())
print("Smallest class size:", y.value_counts().min())

X shape: (18911, 18)
y classes: 10
Smallest class size: 7
